<a href="https://colab.research.google.com/github/saketpab/Vero_Voice_Router_AI_Agent/blob/main/vero_voice_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
#%pip install -U elevenlabs langchain-google-genai langgraph python-dotenv langchain_openai


In [10]:
#%pip install -U vllm

In [11]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

!vllm serve $MODEL_NAME \
    --dtype half \
    --gpu-memory-utilization 0.75 \
    --max-model-len 2048 \
    > /content/vllm.log 2>&1 &

When you try making this in the app, have it route to the csv files with the different jobs, and when u do that u dont need a new node for each service type

Note: Originally created locally, exported to Google COllab to host local model with vLLM

In [12]:
from elevenlabs.client import ElevenLabs
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
from elevenlabs.client import ElevenLabs
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage
from typing import NotRequired, TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import SystemMessage, HumanMessage



In [13]:
#loading api keys
from google.colab import userdata


# load_dotenv()

# print("Gemini key loaded:", bool(os.getenv("GOOGLE_API_KEY")))

client = ElevenLabs(
    api_key=userdata.get('ELEVENLABS_API_KEY')
)

# os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
# model = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash",
#     temperature=0,
# )

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model=MODEL_NAME,
    base_url="http://127.0.0.1:8000/v1",
    api_key="not-needed-for-local-colab",
    temperature=0,
)

In [14]:
response = model.invoke("Reply with exactly: vLLM is working")
print(response.content)

vLLM is working.


In [15]:
#select input audio file
from google.colab import files

uploaded = files.upload()
audio_filename = next(iter(uploaded))

print(audio_filename)

Saving Recording.m4a to Recording.m4a
Recording.m4a


In [16]:
#reading user voice
with open(audio_filename, "rb") as audio:
    response = client.speech_to_text.convert(
        file=audio,
        model_id="scribe_v2"
    )

print(response.text)


My outlet is currently not working. When I try to plug something in, electricity is not coming out of it


In [17]:
#classes:

class VoiceRoutingState(TypedDict):
    transcript: str
    trade: str | None
    service_group_id: str | None
    issue_id: str | None

#first level
class TradeRoute(BaseModel):
    """The first AI decision: which Vero trade should handle this request?"""

    trade: Literal[
        "electricians",
        "plumbers",
        "hvac",
        "lawn-care",
        "handyman",
        "vehicle",
    ] = Field(
        description="The single Vero trade that best matches the customer's spoken problem."
    )

#second level
class ElectricianServiceGroup(BaseModel):
    service_group_id: Literal[
        "outlets-switches",
        "lights-fixtures",
        "breaker-panel-power",
        "wiring-safety",
        "install-new",
        "other-not-sure",
    ] = Field(
        description="The electrician service group that best matches the problem."
    )

#third level
class PowerProblemsIssueRoute(BaseModel):
    issue_id: Literal[
        "elec-no-power-entire-home",
        "elec-no-power-part-home",
        "elec-power-on-off",
        "elec-lights-flickering",
    ] = Field(
        description="The specific electrical power problem."
    )


class OutletsAndSwitchesIssueRoute(BaseModel):
    issue_id: Literal[
        "elec-outlet-not-working",
        "elec-outlet-loose",
        "elec-outlet-sparking",
        "elec-need-new-outlet",
        "elec-outlet-unsafe",
    ] = Field(
        description="The specific outlet or switch problem."
    )


class LightingAndFansIssueRoute(BaseModel):
    issue_id: Literal[
        "elec-light-not-on",
        "elec-light-flickering-dim",
        "elec-need-new-light",
        "elec-fan-not-working",
        "elec-install-fan",
    ] = Field(
        description="The specific lighting or ceiling-fan problem."
    )


class HomeElectricalUpgradesIssueRoute(BaseModel):
    issue_id: Literal[
        "elec-need-more-power",
        "elec-add-outlet-circuit",
        "elec-upgrade-system",
        "elec-fix-inspect-wiring",
        "elec-renovation-work",
    ] = Field(
        description="The specific electrical upgrade or wiring-work request."
    )


class ElectricalInstallationsIssueRoute(BaseModel):
    issue_id: Literal[
        "elec-install-ev-charger",
        "elec-install-appliance-wiring",
        "elec-install-smart-home",
        "elec-install-doorbell-security",
        "elec-outdoor-garage-install",
    ] = Field(
        description="The specific electrical installation request."
    )


class ElectricalSafetyIssueRoute(BaseModel):
    issue_id: Literal[
        "elec-burning-smell",
        "elec-sparks-shock",
        "elec-exposed-wires",
        "elec-buzzing-crackling",
        "elec-emergency-hazard",
    ] = Field(
        description="The specific electrical safety concern."
    )

In [18]:
#nodes
trade_router = model.with_structured_output(TradeRoute)

def route_trade(state: VoiceRoutingState):
    result = trade_router.invoke([
        SystemMessage(content="""
Classify the customer's spoken service request into one trade.

Choices:
- electricians
- plumbers
- hvac
- lawn-care
- handyman
- vehicle
"""),
        HumanMessage(content=state["transcript"]),
    ])

    return {
        "trade": result.trade
    }

electrician_group_router = model.with_structured_output(ElectricianServiceGroup)


def route_electrician_service_group(state: VoiceRoutingState):
    result = electrician_group_router.invoke([
        SystemMessage(content="""

Choose exactly one service group that describes issue:
- outlets-switches: power problems
- lights-fixtures: outlets and switches
- breaker-panel-power: lighting and fans
- wiring-safety: electrical upgrades and wiring work
- install-new: electrical installations
- other-not-sure: electrical safety issues
"""),
        HumanMessage(content=state["transcript"]),
    ])

    return {
        "service_group_id": result.service_group_id
    }

ISSUE_MODEL_BY_SERVICE_GROUP = {
    "outlets-switches": PowerProblemsIssueRoute,
    "lights-fixtures": OutletsAndSwitchesIssueRoute,
    "breaker-panel-power": LightingAndFansIssueRoute,
    "wiring-safety": HomeElectricalUpgradesIssueRoute,
    "install-new": ElectricalInstallationsIssueRoute,
    "other-not-sure": ElectricalSafetyIssueRoute,
}

def route_electrician_issue(state: VoiceRoutingState):
    selected_group = state["service_group_id"]

    issue_output_model = ISSUE_MODEL_BY_SERVICE_GROUP[selected_group]
    issue_router = model.with_structured_output(issue_output_model)

    result = issue_router.invoke([
        SystemMessage(content="""
Choose the issue that best fits the transcript.

For this service group, the choices are:
- elec-outlet-not-working: outlet or switch not working
- elec-outlet-loose: outlet is loose or plug will not stay in
- elec-outlet-sparking: outlet is sparking, buzzing, or smells burnt
- elec-need-new-outlet: customer needs a new outlet or switch installed
- elec-outlet-unsafe: outlet or switch feels unsafe or damaged
"""),
        HumanMessage(content=state["transcript"]),
    ])

    return {"issue_id": result.issue_id}

In [19]:
#conditional edges

def route_after_trade(state: VoiceRoutingState):
    if state["trade"] == "electricians":
        return "route_electrician_service_group"

    return END

In [20]:
builder = StateGraph(VoiceRoutingState)

builder.add_node("route_trade", route_trade)
builder.add_node("route_electrician_service_group",route_electrician_service_group,)
builder.add_node("route_electrician_issue",route_electrician_issue,)


builder.add_edge(START, "route_trade")
builder.add_edge("route_electrician_service_group",END)
builder.add_edge("route_electrician_service_group","route_electrician_issue",)

builder.add_edge("route_electrician_issue",END)

builder.add_conditional_edges("route_trade",route_after_trade)

voice_routing_graph = builder.compile()

In [21]:
result = voice_routing_graph.invoke({
    "transcript": response.text,
    "trade": "",
    "service_group_id": "",
    "issue_id": "",
})

print(result)

{'transcript': 'My outlet is currently not working. When I try to plug something in, electricity is not coming out of it', 'trade': 'electricians', 'service_group_id': 'outlets-switches', 'issue_id': 'elec-no-power-entire-home'}
